<a href="https://colab.research.google.com/github/christinChris/gaiascope/blob/main/notebooks/DroughtScope_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q solara

import ee, geemap
from google.colab import auth, output

auth.authenticate_user()
ee.Initialize(project="ee-christilekka")
output.enable_custom_widget_manager()
print("Connected")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.4/138.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.5/277.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.9/71.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 67.7 MB/s eta 0:00:00


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Connected


In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- Land cover ----------
WORLDCOVER = ee.ImageCollection("ESA/WorldCover/v200").first()

classes = {10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
           50: "Built-up", 60: "Bare / sparse", 70: "Snow and ice",
           80: "Permanent water", 90: "Herbaceous wetland", 95: "Mangroves",
           100: "Moss and lichen"}

LC_COLORS = {10: "#006400", 20: "#FFBB22", 30: "#FFFF4C", 40: "#F096FF",
             50: "#FA0000", 60: "#B4B4B4", 70: "#F0F0F0", 80: "#0064C8",
             90: "#0096A0", 95: "#00CF75", 100: "#FAE6A0"}

NAME_COLORS = {classes[k]: v for k, v in LC_COLORS.items()}


# ---------- Tables and charts ----------
def color_cell(name):
    c = NAME_COLORS.get(name)
    return f"border-left: 14px solid {c}; padding-left: 8px" if c else ""


def lc_table(groups):
    rows = [{"Land cover": classes[g["class"]], "Area (km²)": g["sum"] / 1e6} for g in groups]
    df = pd.DataFrame(rows)
    df = df[df["Area (km²)"] >= 0.05]
    df = df.sort_values("Area (km²)", ascending=False).reset_index(drop=True)
    df["Share (%)"] = df["Area (km²)"] / df["Area (km²)"].sum() * 100
    df.loc[len(df)] = ["Total", df["Area (km²)"].sum(), 100.0]
    return (df.style.format(precision=1)
                    .hide(axis="index")
                    .map(color_cell, subset=["Land cover"]))


def lc_chart(groups, xlabel):
    data = [(g["class"], g["sum"] / 1e6) for g in groups if g["sum"] / 1e6 >= 0.05]
    data = sorted(data, key=lambda item: item[1])
    names  = [classes[c] for c, km2 in data]
    values = [km2 for c, km2 in data]
    colors = [LC_COLORS[c] for c, km2 in data]

    fig, ax = plt.subplots(figsize=(6, 3))
    ax.barh(names, values, color=colors, edgecolor="#555555", linewidth=0.5)
    for i, v in enumerate(values):
        ax.text(v, i, f" {v:.1f}", va="center", fontsize=8)
    ax.set_xlabel(xlabel)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    plt.close(fig)
    return fig


def triangle_chart(r):
    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.scatter(r["sample_ts"], r["sample_ndvi"], s=3, alpha=0.3, color="#52615F", label="Pixels")
    ys = np.linspace(0, max(r["sample_ndvi"]), 50)
    ax.plot(r["a"] + r["b"] * ys, ys, color="#A16207", lw=2, label="Dry edge")
    ax.axvline(r["ts_min"], color="#0369A1", lw=2, label="Wet edge")
    ax.set_xlabel("Ts (°C)")
    ax.set_ylabel("NDVI")
    ax.legend(frameon=False, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    plt.close(fig)
    return fig


# ---------- Landsat 8 + 9: Ts and NDVI ----------
def landsat_prep(img):
    qa = img.select("QA_PIXEL")
    clear = (qa.bitwiseAnd(1 << 1).eq(0)        # dilated cloud
             .And(qa.bitwiseAnd(1 << 3).eq(0))  # cloud
             .And(qa.bitwiseAnd(1 << 4).eq(0))  # cloud shadow
             .And(qa.bitwiseAnd(1 << 5).eq(0))) # snow
    sr   = img.select(["SR_B2", "SR_B3", "SR_B4", "SR_B5"]).multiply(0.0000275).add(-0.2)
    ndvi = sr.normalizedDifference(["SR_B5", "SR_B4"]).rename("NDVI")
    ts   = img.select("ST_B10").multiply(0.00341802).add(149.0).subtract(273.15).rename("Ts")
    return sr.addBands([ndvi, ts]).updateMask(clear)


def landsat_collection(start, end, area, max_cloud=60):
    return (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
            .merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2"))
            .filterBounds(area)
            .filterDate(start, end)
            .filter(ee.Filter.lt("CLOUD_COVER", max_cloud))
            .map(landsat_prep))


# ---------- DroughtScope science ----------
def droughtscope(aoi, dates, tvdi_thr=0.7, bin_width=0.05, min_count=50):
    col  = landsat_collection(dates[0], dates[1], aoi)
    img  = col.median().clip(aoi)
    lc   = WORLDCOVER.clip(aoi)
    land = lc.neq(80).And(img.select("NDVI").gt(0))
    ndvi = img.select("NDVI").updateMask(land)
    ts   = img.select("Ts").updateMask(land)

    # 1. Ts per NDVI bin (1st and 99th percentile) + a pixel sample for the plot
    bins = ndvi.divide(bin_width).floor().int().rename("bin")
    reducer = (ee.Reducer.percentile([1, 99])
               .combine(ee.Reducer.count(), sharedInputs=True)
               .group(groupField=1, groupName="bin"))
    stats = (ts.addBands(bins)
             .reduceRegion(reducer=reducer, geometry=aoi, scale=60, maxPixels=1e10)
             .get("groups"))
    sample = ndvi.addBands(ts).sample(region=aoi, scale=60, numPixels=3000, seed=1)
    info = ee.Dictionary({"stats": stats, "n": col.size(),
                          "ndvi": sample.aggregate_array("NDVI"),
                          "ts": sample.aggregate_array("Ts")}).getInfo()

    # 2. Fit the dry edge (Python), wet edge = minimum Ts
    rows = [g for g in info["stats"] if g["count"] >= min_count]
    x    = [(g["bin"] + 0.5) * bin_width for g in rows]
    hot  = [g["p99"] for g in rows]
    cool = [g["p1"] for g in rows]
    fit_x = [xi for xi in x if xi >= 0.15]
    fit_y = [hot[i] for i, xi in enumerate(x) if xi >= 0.15]
    if len(fit_x) < 3:
        raise ValueError("Not enough NDVI variety to build the triangle. "
                         "Draw a bigger box with both bare and green land.")
    b, a   = np.polyfit(fit_x, fit_y, 1)
    ts_min = min(cool)

    # 3. TVDI and dry mask (server)
    dry_edge = ndvi.multiply(b).add(a)
    tvdi = ts.subtract(ts_min).divide(dry_edge.subtract(ts_min)).clamp(0, 1).rename("TVDI")
    dry  = tvdi.gt(tvdi_thr).rename("dry")

    # 4. Numbers (one trip)
    area   = ee.Image.pixelArea().updateMask(dry)
    groups = (area.addBands(lc)
              .reduceRegion(reducer=ee.Reducer.sum().group(groupField=1, groupName="class"),
                            geometry=aoi, scale=30, maxPixels=1e10)
              .get("groups"))
    land_m2 = (ee.Image.pixelArea().updateMask(land)
               .reduceRegion(reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e10)
               .get("area"))
    mean_tvdi = tvdi.reduceRegion(reducer=ee.Reducer.mean(), geometry=aoi,
                                  scale=60, maxPixels=1e10).get("TVDI")
    info2 = ee.Dictionary({"groups": groups, "land": land_m2, "mean": mean_tvdi}).getInfo()

    dry_km2  = sum(g["sum"] for g in info2["groups"]) / 1e6
    land_km2 = (info2["land"] or 0) / 1e6

    return {"image": img, "tvdi": tvdi, "dry": dry, "n": info["n"],
            "a": a, "b": b, "ts_min": ts_min,
            "sample_ndvi": info["ndvi"], "sample_ts": info["ts"],
            "mean_tvdi": info2["mean"], "km2": dry_km2,
            "share": 100 * dry_km2 / land_km2 if land_km2 else 0,
            "groups": info2["groups"],
            "table": lc_table(info2["groups"]) if info2["groups"] else None}

print("Engine ready")

Engine ready


In [8]:
import solara, ipyleaflet

# ---------- State ----------
start    = solara.reactive("2024-07-01")
end      = solara.reactive("2024-08-31")
tvdi_thr = solara.reactive(0.7)
basemap  = solara.reactive("Google satellite")
area     = solara.reactive(ee.Geometry.Rectangle([21.8, 39.25, 22.9, 39.75]))
result   = solara.reactive(None)
status   = solara.reactive("")

BASEMAPS = {
    "Google satellite": "https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
    "Google map":       "https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
    "Google terrain":   "https://mt1.google.com/vt/lyrs=p&x={x}&y={y}&z={z}",
}
TRUE_COLOR   = {"bands": ["SR_B4", "SR_B3", "SR_B2"], "min": 0, "max": 0.3}
TVDI_PALETTE = ["#2166AC", "#67A9CF", "#F7F7F7", "#FDB863", "#B2182B"]
TVDI_VIS     = {"min": 0, "max": 1, "palette": TVDI_PALETTE}
RESULT_LAYERS = ("Landsat · true color", "TVDI", "Study area")

# ---------- Map ----------
m = geemap.Map(height="600px")
outline = {"shapeOptions": {"color": "#FFFFFF", "weight": 2, "fillOpacity": 0}}
m.draw_control.rectangle = outline
m.draw_control.polygon = outline
base = m.layers[0]
base.url = BASEMAPS[basemap.value]
base.name = "Basemap"
base.attribution = "Imagery © Google"
m.centerObject(area.value, 9)
m.add_colorbar(TVDI_VIS, label="TVDI (0 = wet, 1 = dry)", position="bottomright")


# ---------- Map helpers ----------
def set_basemap(name):
    base.url = BASEMAPS[name]


def remove_results():
    for c in list(m.controls):
        if isinstance(c, ipyleaflet.SplitMapControl):
            m.remove_control(c)
    for layer in list(m.layers):
        if layer.name in RESULT_LAYERS:
            m.remove_layer(layer)


def update_map(*args):
    r = result.value
    if r is None:
        return
    remove_results()
    left  = geemap.ee_tile_layer(r["image"], TRUE_COLOR, "Landsat · true color")
    right = geemap.ee_tile_layer(r["tvdi"],  TVDI_VIS,   "TVDI")
    m.add_layer(left)
    m.add_layer(right)
    m.add_control(ipyleaflet.SplitMapControl(left_layer=left, right_layer=right))
    m.addLayer(ee.Image().paint(area.value, 0, 2), {"palette": ["#FFFFFF"]}, "Study area")


# ---------- Actions ----------
def run():
    status.value = ""
    try:
        if m.user_roi is not None:
            area.value = m.user_roi
            m.draw_control.clear()
        result.value = droughtscope(area.value, (start.value, end.value), tvdi_thr.value)
        update_map()
        m.centerObject(area.value)
    except Exception as e:
        status.value = f"⚠️ {e}"


def clear_area(*args):
    remove_results()
    m.draw_control.clear()
    result.value = None


# ---------- Page ----------
@solara.component
def Page():
    with solara.Columns([1, 3]):
        with solara.Column():
            solara.Markdown("# ☀️ DroughtScope")
            solara.Info("Draw a box with both bare and green land, so the Ts/VI triangle has real edges.")
            solara.Markdown("**When**")
            solara.InputText("From", value=start)
            solara.InputText("To", value=end)
            solara.SliderFloat("Dry if TVDI above", value=tvdi_thr,
                               min=0.4, max=0.95, step=0.05, thumb_label="always")
            solara.Button("Map dryness", on_click=run, color="primary")
            solara.Button("New area", on_click=clear_area, outlined=True)
            solara.Markdown(status.value)

            solara.Markdown("**Map layers**")
            solara.Select("Basemap", value=basemap, values=list(BASEMAPS), on_value=set_basemap)

            r = result.value
            if r is not None:
                solara.Markdown(f"Landsat scenes: **{r['n']}** · Mean TVDI: **{r['mean_tvdi']:.2f}**")
                solara.Markdown(f"### ☀️ Dry area: {r['km2']:.1f} km² ({r['share']:.0f}% of land)")
                solara.Markdown("**Ts/VI space**")
                solara.FigureMatplotlib(triangle_chart(r))
                solara.Markdown(f"Dry edge: Ts = {r['a']:.1f} {r['b']:+.1f}·NDVI · Wet edge: Ts = {r['ts_min']:.1f} °C")
                if r["table"] is not None:
                    solara.display(r["table"])
                    solara.Markdown("**Dry area by land cover**")
                    solara.FigureMatplotlib(lc_chart(r["groups"], "Dry area (km²)"))
        with solara.Column():
            solara.display(m)
Page()

Html(layout=None, style_='display: none', tag='span')

Cannot show ipywidgets in text

In [9]:
m

Map(center=[39.5, 22.35], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…